# 🚀 Startup Metrics Dashboard — Python Analysis Notebook
**Author:** Pretty | MCA Final Year  
**Tools:** Python · Pandas · Plotly · SQLite  
**Dataset:** Simulated SaaS startup user events (2,000 users, 90 days)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
users  = pd.read_csv('../data/users.csv',   parse_dates=['signup_date'])
events = pd.read_csv('../data/events.csv',  parse_dates=['event_date','event_ts'])
sessions = pd.read_csv('../data/sessions.csv')

print(f'Users:    {len(users):,}')
print(f'Events:   {len(events):,}')
print(f'Sessions: {len(sessions):,}')
users.head()

## 2. Data Overview & Quality Check

In [ ]:
print('=== Users Schema ===')
print(users.dtypes)
print('\n=== Missing Values ===')
print(users.isnull().sum())
print('\n=== Event Types ===')
print(events['event'].value_counts())
print('\n=== Date Range ===')
print(f"From: {events['event_date'].min().date()} → To: {events['event_date'].max().date()}")

## 3. DAU — Daily Active Users

In [ ]:
dau = events.groupby('event_date')['user_id'].nunique().reset_index()
dau.columns = ['date', 'dau']
dau['rolling7'] = dau['dau'].rolling(7).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dau['date'], y=dau['dau'], fill='tozeroy',
                          fillcolor='rgba(99,102,241,0.1)', line=dict(color='#6366f1', width=2),
                          name='DAU'))
fig.add_trace(go.Scatter(x=dau['date'], y=dau['rolling7'],
                          line=dict(color='#f59e0b', width=2, dash='dot'), name='7-day avg'))
fig.update_layout(title='Daily Active Users', template='plotly_dark')
fig.show()

print(f'Avg DAU (last 30 days): {dau["dau"].tail(30).mean():.0f}')

## 4. MAU — Monthly Active Users

In [ ]:
events['month'] = events['event_date'].dt.to_period('M')
mau = events.groupby('month')['user_id'].nunique().reset_index()
mau.columns = ['month', 'mau']
mau['month_str'] = mau['month'].astype(str)

fig = px.bar(mau, x='month_str', y='mau', color_discrete_sequence=['#8b5cf6'],
             title='Monthly Active Users', template='plotly_dark')
fig.show()

# Stickiness
avg_dau = dau['dau'].tail(30).mean()
latest_mau = mau['mau'].iloc[-1]
stickiness = round(avg_dau / latest_mau * 100, 1)
print(f'MAU: {latest_mau} | Avg DAU: {avg_dau:.0f} | Stickiness: {stickiness}%')

## 5. Conversion Funnel

In [ ]:
funnel_order = ['signup','activation','feature_use','upgrade','paying']
funnel = events.groupby('event')['user_id'].nunique().reindex(funnel_order)

funnel_df = pd.DataFrame({
    'Stage': funnel_order,
    'Users': funnel.values,
    'Pct of Signup': [round(v/funnel.values[0]*100,1) for v in funnel.values]
})
print(funnel_df.to_string(index=False))

fig = go.Figure(go.Funnel(
    y=funnel_df['Stage'], x=funnel_df['Users'],
    textinfo='value+percent initial',
    marker=dict(color=['#6366f1','#7c3aed','#8b5cf6','#a78bfa','#10b981'])
))
fig.update_layout(title='Conversion Funnel', template='plotly_dark')
fig.show()

## 6. Retention — Day 1, Day 7, Day 30

In [ ]:
signup_dates = events[events['event']=='signup'].groupby('user_id')['event_date'].min().rename('signup_date')
activity = events.groupby('user_id')['event_date'].apply(set)

def check_ret(uid, sd, days):
    if uid not in activity.index: return 0
    return int(any((d - sd).days in range(days, days+2) for d in activity[uid]))

ret_df = signup_dates.reset_index()
ret_df['d1']  = ret_df.apply(lambda r: check_ret(r.user_id, r.signup_date,  1), axis=1)
ret_df['d7']  = ret_df.apply(lambda r: check_ret(r.user_id, r.signup_date,  7), axis=1)
ret_df['d30'] = ret_df.apply(lambda r: check_ret(r.user_id, r.signup_date, 30), axis=1)

n = len(ret_df)
print(f'Day-1  Retention: {ret_df["d1"].sum()/n*100:.1f}%')
print(f'Day-7  Retention: {ret_df["d7"].sum()/n*100:.1f}%')
print(f'Day-30 Retention: {ret_df["d30"].sum()/n*100:.1f}%')

fig = px.bar(x=['Day 1','Day 7','Day 30'],
             y=[ret_df['d1'].mean()*100, ret_df['d7'].mean()*100, ret_df['d30'].mean()*100],
             color_discrete_sequence=['#6366f1','#8b5cf6','#a78bfa'],
             labels={'x':'Period','y':'Retention %'}, title='User Retention',
             template='plotly_dark')
fig.show()

## 7. Drop-off Analysis

In [ ]:
vals = funnel.values
drop_df = pd.DataFrame({
    'Step': ['Signup→Activation','Activation→Feature','Feature→Upgrade','Upgrade→Paying'],
    'Dropped': [vals[0]-vals[1], vals[1]-vals[2], vals[2]-vals[3], vals[3]-vals[4]],
    'Drop %':  [round((vals[0]-vals[1])/vals[0]*100,1),
                round((vals[1]-vals[2])/vals[1]*100,1),
                round((vals[2]-vals[3])/vals[2]*100,1),
                round((vals[3]-vals[4])/vals[3]*100,1)]
})
print(drop_df.to_string(index=False))

fig = px.bar(drop_df, x='Step', y='Drop %',
             color='Drop %', color_continuous_scale=['#10b981','#f59e0b','#ef4444'],
             title='Drop-off at Each Funnel Stage', template='plotly_dark')
fig.show()

## 8. North Star Metric — Weekly Active Paying Users

In [ ]:
events['week'] = events['event_date'].dt.to_period('W')
nsm = events[events['event']=='paying'].groupby('week')['user_id'].nunique().reset_index()
nsm.columns = ['week','wapu']
nsm['week_str'] = nsm['week'].astype(str)

print('North Star: Weekly Active Paying Users (WAPU)')
print(nsm[['week_str','wapu']].to_string(index=False))

fig = px.line(nsm, x='week_str', y='wapu', markers=True,
              color_discrete_sequence=['#a78bfa'],
              title='North Star Metric — WAPU', template='plotly_dark')
fig.show()

## 9. Revenue Analysis

In [ ]:
paying = events[events['event']=='paying'].merge(users[['user_id','plan','channel']], on='user_id')

total_rev = paying['revenue'].sum()
aov       = paying['revenue'].mean()
print(f'Total Revenue: ${total_rev:,.2f}')
print(f'Avg Order Value: ${aov:.2f}')
print(f'Paying Users: {paying["user_id"].nunique():,}')

# Revenue by plan
rev_plan = paying.groupby('plan')['revenue'].sum().reset_index()
fig = px.pie(rev_plan, names='plan', values='revenue', hole=0.5,
             title='Revenue by Plan', template='plotly_dark')
fig.show()

# Monthly Revenue
rev_monthly = paying.groupby('month')['revenue'].sum().reset_index()
rev_monthly['ms'] = rev_monthly['month'].astype(str)
fig = px.bar(rev_monthly, x='ms', y='revenue', color_discrete_sequence=['#10b981'],
             title='Monthly Revenue', template='plotly_dark')
fig.show()

## 10. Acquisition Channel Performance

In [ ]:
paying_ids = set(events[events['event']=='paying']['user_id'])
users['is_paying'] = users['user_id'].isin(paying_ids)

ch_perf = users.groupby('channel').agg(
    signups=('user_id','count'),
    paying=('is_paying','sum')
).reset_index()
ch_perf['conv_pct'] = (ch_perf['paying']/ch_perf['signups']*100).round(1)
print(ch_perf.sort_values('conv_pct', ascending=False).to_string(index=False))

fig = px.bar(ch_perf.sort_values('conv_pct'), x='conv_pct', y='channel',
             orientation='h', color='conv_pct',
             color_continuous_scale=['#1e1b4b','#6366f1','#a5b4fc'],
             title='Channel Conversion Rate', template='plotly_dark')
fig.show()

## 11. Business Recommendations

Based on the analysis above:

| # | Finding | Recommendation | Priority |
|---|---------|----------------|----------|
| 1 | Largest drop-off is Signup→Activation | Redesign onboarding to deliver 'aha moment' in <3 min | 🔴 High |
| 2 | Day-1 retention below 30% | Add welcome email + next-step CTA immediately after signup | 🔴 High |
| 3 | Feature→Upgrade drop-off | Show upgrade prompt contextually at premium feature touchpoints | 🟡 Medium |
| 4 | Stickiness < 20% | Introduce daily value loop — digest, streak, or notification | 🟡 Medium |
| 5 | Referral likely best channel | Launch referral program — reward both sides | 🟢 Low |

**North Star focus:** Grow WAPU (Weekly Active Paying Users) by 10% week-over-week.
Every product decision should be evaluated against this metric.